<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/BandOffSet_QE_InterMat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Band offset calcularions with InterMat+QuantumEspresso
(Note: for computational cost reasons, in this demo we use less k-points)



In [ ]:
%%time
import os
os.chdir('/content')
# Clone the QE git
if not os.path.exists('q-e'):
   os.system('git clone https://github.com/QEF/q-e.git')
# Install the dependencies
!apt-get install -y libfftw3-3 libfftw3-dev libfftw3-doc
os.chdir('q-e')
# run the configure file
#!DFLAGS='-D__OPENMP -D__FFTW3 -D__MPI -D__SCALAPACK' FFT_LIBS='-lfftw3'  ./configure --enable-openmp

!!DFLAGS='-D__FFTW3 ' FFT_LIBS='-lfftw3'  ./configure
! make pw

In [ ]:
!make pp

In [ ]:
!pip install -qqq intermat spglib ase

In [ ]:
import os
os.chdir('/content')

In [ ]:
from jarvis.core.atoms import Atoms
from jarvis.io.vasp.inputs import Poscar
from intermat.generate import InterfaceCombi
import numpy as np


film_pos = """Si2
1.0
3.3641499856336465 -2.5027128e-09 1.94229273881412
1.121382991333525 3.1717517190189715 1.9422927388141193
-2.5909987e-09 -1.8321133e-09 3.884586486670313
Si
2
Cartesian
3.92483875 2.77528125 6.7980237500000005
0.56069125 0.39646875 0.9711462500000001
"""
subs_pos = """GaAs
1.0
3.509895098013499 -6.584084e-10 2.026439508259945
1.169965032050413 3.3091606691404545 2.026439508259945
-1.8622597e-09 -1.3168164e-09 4.052878022970947
Ga As
1 1
Cartesian
0.0 0.0 0.0
1.1699675 0.82729 2.02644
"""



film = Poscar.from_string(film_pos).atoms
subs = Poscar.from_string(subs_pos).atoms

x = InterfaceCombi(
    film_mats=[film],
    subs_mats=[subs],
    film_indices=[[1,1,0]],
    subs_indices=[[1,1,0]],

    disp_intvl=0.1,
    vacuum_interface=2,
    # dataset=None,
    from_conventional_structure_film=True,
    from_conventional_structure_subs=True,
)
wads = x.calculate_wad(method="ewald")
index = np.argmin(wads) # important for disp_intvl not equal zero
combined = Atoms.from_dict(x.generated_interfaces[index]["generated_interface"])
film_interface = Atoms.from_dict(x.generated_interfaces[index]["film_surf"])
subs_interface = Atoms.from_dict(x.generated_interfaces[index]["subs_surf"])


In [ ]:
from ase.visualize import view
%matplotlib inline
view(combined.ase_converter(),viewer='x3d')

In [ ]:
from ase.visualize import view
view(film_interface.ase_converter(),viewer='x3d')

In [ ]:
from ase.visualize import view
view(subs_interface.ase_converter(),viewer='x3d')

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=film_interface,
    extra_params=params,
    jobname="film",
)
out_film= calc.predict()
en = out_film["total_energy"]
print(en)

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=subs_interface,
    extra_params=params,
    jobname="subs",
)
out_subs= calc.predict()
en = out_subs["total_energy"]
print(en)

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=combined,
    extra_params=params,
    jobname="interface",
)
out_int= calc.predict()
en = out_int["total_energy"]
print(en)

Bulk calculations for getting DeltaE

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=film,
    extra_params=params,
    jobname="bulk_film",
)
out_int= calc.predict()
en = out_int["total_energy"]

from jarvis.io.qe.outputs import DataFileSchema

d=DataFileSchema("bulk_film/RELAX.save/data-file-schema.xml")
efermi_bulk_film = d.efermi
print('efermi_bulk_film',efermi_bulk_film)

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=subs,
    extra_params=params,
    jobname="bulk_subs",
)
out_int= calc.predict()
en = out_int["total_energy"]

from jarvis.io.qe.outputs import DataFileSchema

d=DataFileSchema("bulk_subs/RELAX.save/data-file-schema.xml")
efermi_bulk_subs = d.efermi
print('efermi_bulk_subs',efermi_bulk_subs)

In [ ]:
delta_E=efermi_bulk_film-efermi_bulk_subs
print(delta_E)

In [ ]:
!rm avg.dat
lines="""&inputPP
   outdir='film',
   prefix='RELAX',
   plot_num=11,
   filplot = 'film.pot'
/

&plot
   filepp(1) = 'film.pot'
   iflag=3,
   output_format=5,
   fileout = 'film.xsf',
/
"""
with open("pp_film.in","w") as f:
    f.write(lines)
!/content/q-e/bin/pp.x<pp_film.in
lines="""1
film.pot
1.0
25
3
2.0
"""
with open("average_film.in","w") as f:
    f.write(lines)
!/content/q-e/bin/average.x<average_film.in
import numpy as np
%matplotlib inline
import  matplotlib.pyplot as plt

film_avg=13.6*np.loadtxt('avg.dat')
from jarvis.io.qe.outputs import DataFileSchema

d=DataFileSchema("film/RELAX.save/data-file-schema.xml")
efermi_film = d.efermi
plt.plot(film_avg[:,0],film_avg[:,1])
plt.axhline(y=efermi_film,c='g')
plt.axhline(y=max(film_avg[:,1]),c='orange')
wf=round(max(film_avg[:,1])-efermi_film,3)
plt.title('Work function: '+str(wf)+'eV')
plt.show()

In [ ]:
!rm avg.dat
lines="""&inputPP
   outdir='subs',
   prefix='RELAX',
   plot_num=11,
   filplot = 'subs.pot'
/

&plot
   filepp(1) = 'subs.pot'
   iflag=3,
   output_format=5,
   fileout = 'subs.xsf',
/
"""
with open("pp_subs.in","w") as f:
    f.write(lines)
!/content/q-e/bin/pp.x<pp_subs.in
lines="""1
subs.pot
1.0
25
3
2.0
"""
with open("average_subs.in","w") as f:
    f.write(lines)
!/content/q-e/bin/average.x<average_subs.in
import numpy as np
%matplotlib inline
import  matplotlib.pyplot as plt

subs_avg=13.6*np.loadtxt('avg.dat')
from jarvis.io.qe.outputs import DataFileSchema

d=DataFileSchema("subs/RELAX.save/data-file-schema.xml")
efermi_subs = d.efermi
plt.plot(subs_avg[:,0],subs_avg[:,1])
plt.axhline(y=efermi_subs,c='g')
plt.axhline(y=max(subs_avg[:,1]),c='orange')
wf=round(max(subs_avg[:,1])-efermi_subs,3)
plt.title('Work function: '+str(wf)+'eV')
plt.show()

In [ ]:
!rm avg.dat
lines="""&inputPP
   outdir='interface',
   prefix='RELAX',
   plot_num=11,
   filplot = 'interface.pot'
/

&plot
   filepp(1) = 'interface.pot'
   iflag=3,
   output_format=5,
   fileout = 'interface.xsf',
/
"""
with open("pp_interface.in","w") as f:
    f.write(lines)
!/content/q-e/bin/pp.x<pp_interface.in
lines="""1
interface.pot
1.0
25
3
2.0
"""
with open("average_interface.in","w") as f:
    f.write(lines)
!/content/q-e/bin/average.x<average_interface.in
import numpy as np
%matplotlib inline
import  matplotlib.pyplot as plt

interface_avg=13.6*np.loadtxt('avg.dat')
from jarvis.io.qe.outputs import DataFileSchema

d=DataFileSchema("interface/RELAX.save/data-file-schema.xml")
efermi_interface = d.efermi
plt.plot(interface_avg[:,0],interface_avg[:,1])
plt.axhline(y=efermi_interface,c='g')
plt.axhline(y=max(interface_avg[:,1]),c='orange')
wf=round(max(interface_avg[:,1])-efermi_interface,3)
# plt.title('Work function: '+str(wf)+'eV')
plt.show()


In [ ]:
from scipy.interpolate import CubicSpline
import scipy.integrate as integrate
from numpy import ones, vstack
from numpy.linalg import lstsq
from scipy.signal import find_peaks

step_size = 10

def get_best_L(start_L, end_L, S, x_target):
    """Find best L on grid of 10 points."""
    L = 0
    best = 1000000000.0
    for L_guess in np.arange(
        start_L, end_L + 1e-5, (end_L - start_L) / (step_size + 0.55)
    ):
        current = 0.0
        for xx in x_target:
            current += abs(S(xx) - S(xx + L_guess))
        if current < best:
            best = current
            L = L_guess

    return L

def best_L_recursive(start_L, end_L, S, x_target):
    """Find recursively best L."""
    L_best = 0.0
    L_range = end_L - start_L
    for iter in range(step_size):
        L_best = get_best_L(start_L, end_L, S, x_target)
        L_range = L_range / step_size
        start_L = L_best - L_range
        end_L = L_best + L_range

    return L_best

def get_m_c(x=[], y=[]):
    """Get least sq. fit."""
    A = vstack([x, ones(len(x))]).T
    m, c = lstsq(A, y)[0]
    return m, c

def get_mean_val(x_target, XX, AVG):
    """Get man val."""
    x_target = np.array(x_target)
    XX = np.array(XX)
    AVG = np.array(AVG)
    new_x = XX.searchsorted(x_target)
    new_mean = np.mean(AVG[new_x])
    # print('new_mean',new_mean)
    m, c = get_m_c(x=XX[new_x], y=AVG[new_x])
    # print('new_x',new_x)
    # print('AVG[new_x]',AVG[new_x])
    # print('m,c',m,c)
    return new_mean, m, c

def do_average(L, x, S):
    """Do the actual averaging."""
    AVG = []
    XX = []
    for xx in x:
        if xx - L / 2.0 < x[0]:
            continue
        if xx + L / 2.0 > x[-1]:
            continue
        XX.append(xx)

        # tmp_x = np.arange(xx - L / 2.0, xx + L / 2.0, 0.1)
        # x=np.arange(xx-L/2.0, xx+L/2.0,0.1)
        # tmp_y = S(tmp_x)
        # intg2=np.trapz(tmp_y,tmp_x,tmp_x[1]-tmp_x[0])/L
        intg2 = integrate.quad(S, xx - L / 2.0, xx + L / 2.0)[0] / L
        # print('tmp_x[1]-tmp_x[0]',tmp_x[1]-tmp_x[0])
        # intg2=integrate.simpson(tmp_y,tmp_x,tmp_x[1]-tmp_x[0]) / L
        # print('intg',int)
        AVG.append(intg2)  # integration
    #        print("int ", integrate.quad(S, xx-L/2.0, xx+L/2.0))
    return XX, AVG

def offset(x=[], s=[], width=5, left_index=-1, polar=False,deltaE=0):
    """Get valence band offset."""

    S = CubicSpline(x, s)

    max_peaks, properties = find_peaks(s, prominence=1, width=width)
    max_peaks = max_peaks[:-1]

    print("Number of peaks ", len(max_peaks))
    if left_index == -1:  # automatically pick left_index from max_peaks
        print("auto detect left index")
        if len(max_peaks) <= 8:
            print("WARNING, not many peaks found")
            left_index = 1
        elif len(max_peaks) <= 12:
            left_index = 2
        else:
            left_index = 3
    else:
        print("use input left index")
    print("left index ", left_index)
    right_index = left_index * -1 + 1

    plt.plot(x[max_peaks], s[max_peaks], "x")
    # tmp=int((max_peaks[left_index]-max_peaks[left_index+2]))
    x_target1 = x[
        np.arange(max_peaks[left_index], max_peaks[left_index + 1], 2)
    ]

    # initial guess left
    L_guess_peaks_left = x_target1[-1] - x_target1[0]

    # points in left cell
    # x_target1 = x[ range(50, 100,2) ]
    print("Initial guess left ", L_guess_peaks_left)
    L = best_L_recursive(1.0, L_guess_peaks_left * 1.5, S, x_target1)
    print("Lleft ", L)

    plt.plot(x, s, c="k")
    XX, AVG = do_average(L, x, S)

    meanval1, m1, c1 = get_mean_val(x_target1, XX, AVG)
    if polar:
        plt.plot(XX, np.array(XX) * m1 + c1, c="purple", linestyle="-.")
    # print('x_target1',x_target1)
    # print('XX',XX)
    # print('AVG',AVG)
    plt.plot(XX, AVG, c="r")

    # x_target2 = x[-200:-150]
    # tmp=int((max_peaks[right_index-1]-max_peaks[right_index]))
    x_target2 = x[
        np.arange(max_peaks[right_index - 1], max_peaks[right_index], 2)
    ]
    # x_target2 = x[ range(-100, ,2) ]

    # initial guess right
    L_guess_peaks_right = x_target2[-1] - x_target2[0]
    L = best_L_recursive(1.0, L_guess_peaks_right * 1.5, S, x_target2)
    print("Initial guess right ", L_guess_peaks_right)
    print("Lright ", L)

    plt.plot(x, s, c="k")
    XX, AVG = do_average(L, x, S)
    meanval2, m2, c2 = get_mean_val(x_target2, XX, AVG)
    if polar:
        plt.plot(XX, np.array(XX) * m2 + c2, c="orange", linestyle="-.")

    if polar:
        mid_point = int((len(XX) - 1) / 2)
        polar_del_V = (np.array(XX) * m2 + c2)[mid_point] - (
            np.array(XX) * m1 + c1
        )[mid_point]
        plt.plot(mid_point, -7, "*")
        plt.axvline(x=XX[mid_point], linestyle="-.", c="blue")
        print(
            "polar delV,mid_point",
            polar_del_V,
            mid_point,
            len(XX),
            XX[mid_point],
        )

    plt.plot(XX, AVG, c="g")
    plt.plot(x_target1, S(x_target1), "c")
    plt.plot(x_target2, S(x_target2), "c")
    deltaV = meanval2 - meanval1
    deltaV = deltaV/hartree_to_ev
    phi = deltaV + deltaE
    if polar:
        phi = polar_del_V + deltaE
        deltaV = polar_del_V
    plt.grid(color="gray", ls="-.")
    plt.minorticks_on()
    plt.ylabel("Potential (eV)")
    plt.xlim(0, max(x))
    plt.xlabel(r"Distance ($\AA$)")
    # plt.plot(x,[meanval1 for i in range(len(x))],linestyle='-.',color='blue')
    # plt.plot(x,[meanval2 for i in range(len(x))],linestyle='-.',color='blue')
    print("meanval ", [meanval1, meanval2], meanval2 - meanval1, phi)
    plt.title("Offset (eV): " + str(round(phi, 2)))
    filename = "offset.png"
    print("deltaE", deltaE)
    plt.show()
    # plt.savefig(filename)
    # plt.close()
    info = {}
    info["phi"] = phi
    info["polar"] = polar
    info["deltaV"] = deltaV
    info["deltaE"] = deltaE
    info["L_guess_peaks_right"] = L_guess_peaks_right
    info["L"] = L
    info["left_index"] = left_index

    return info  # phi
hartree_to_ev=27.2114 #need to check
offset(x=interface_avg[:,0],s=np.array(interface_avg[:,1]),deltaE=delta_E)

In [ ]:
6.23/.23

Extra blocks

In [ ]:
from jarvis.core.atoms import Atoms
from jarvis.io.vasp.inputs import Poscar
from intermat.generate import InterfaceCombi
import numpy as np
film_pos = """MoS2
1.0
1.5939818713880687 -2.7608568870413697 0.0
1.5939818713880687 2.7608568870413697 0.0
0.0 0.0 34.879004
Mo S
1 2
Cartesian
1.59398 -0.9202885072399822 3.7197407130000837
1.59398 0.9202885072399822 2.153121547584768
1.59398 0.9202885072399822 5.286394757415309
"""
subs_pos = """Te2Mo
1.0
1.7811816828576317 -3.08509790977307 0.0
1.7811816828576317 3.08509790977307 0.0
0.0 0.0 35.346645
Te Mo
2 1
Cartesian
1.78118 -1.02836872339998 2.024632306765125
1.78118 -1.02836872339998 5.648699077555013
1.78118 1.0283687233999803 3.8366656921600706
"""

# film_pos = """Si2
# 1.0
# 3.3641499856336465 -2.5027128e-09 1.94229273881412
# 1.121382991333525 3.1717517190189715 1.9422927388141193
# -2.5909987e-09 -1.8321133e-09 3.884586486670313
# Si
# 2
# Cartesian
# 3.92483875 2.77528125 6.7980237500000005
# 0.56069125 0.39646875 0.9711462500000001
# """
# subs_pos = """GaAs
# 1.0
# 3.509895098013499 -6.584084e-10 2.026439508259945
# 1.169965032050413 3.3091606691404545 2.026439508259945
# -1.8622597e-09 -1.3168164e-09 4.052878022970947
# Ga As
# 1 1
# Cartesian
# 0.0 0.0 0.0
# 1.1699675 0.82729 2.02644
# """
# subs_pos="""Ge2
# 1.0
# 3.5261943628070638 1.196611e-10 2.035849771621022
# 1.1753977879818787 3.3245278496612793 2.035849771621022
# -4.62335e-10 -3.269202e-10 4.071698544842951
# Ge
# 2
# Cartesian
# 4.11389125 2.9089637500000003 7.125475
# 0.58769875 0.41556625 1.017925
# """


film = Poscar.from_string(film_pos).atoms
subs = Poscar.from_string(subs_pos).atoms

x = InterfaceCombi(
    film_mats=[film],
    subs_mats=[subs],
    # film_indices=[[1,1,0]],
    # subs_indices=[[1,1,0]],

    film_indices=[[0,0,1]],
    subs_indices=[[0,0,1]],

    disp_intvl=0.0,
    vacuum_interface=2,
    # dataset=None,
    from_conventional_structure_film=False,
    from_conventional_structure_subs=False,
)
wads = x.calculate_wad(method="ewald")
index = np.argmin(wads) # important for disp_intvl not equal zero
combined = Atoms.from_dict(x.generated_interfaces[index]["generated_interface"])
film_interface = Atoms.from_dict(x.generated_interfaces[index]["film_surf"]).center(vacuum=6)
subs_interface = Atoms.from_dict(x.generated_interfaces[index]["subs_surf"]).center(vacuum=6)


In [ ]:
from ase.visualize import view
view(combined.ase_converter(),viewer='x3d')

In [ ]:
from ase.visualize import view
view(film_interface.ase_converter(),viewer='x3d')

In [ ]:
from ase.visualize import view
view(subs_interface.ase_converter(),viewer='x3d')

In [ ]:
combined.num_atoms

In [ ]:
os.chdir('/content')

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=film_interface,
    extra_params=params,
    jobname="film",
)
out_film= calc.predict()
en = out_film["total_energy"]
print(en)

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=subs_interface,
    extra_params=params,
    jobname="subs",
)
out_subs= calc.predict()
en = out_subs["total_energy"]
print(en)

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=combined,
    extra_params=params,
    jobname="interface",
)
out_int= calc.predict()
en = out_int["total_energy"]
print(en)

In [ ]:
os.chdir('/content')

### Post-processing

In [ ]:
lines="""&inputPP
   outdir='film',
   prefix='RELAX',
   plot_num=11,
   filplot = 'film.pot'
/

&plot
   filepp(1) = 'film.pot'
   iflag=3,
   output_format=5,
   fileout = 'film.xsf',
/
"""
with open("pp_film.in","w") as f:
    f.write(lines)
!/content/q-e/bin/pp.x<pp_film.in

In [ ]:
lines="""1
film.pot
1.0
25
3
2.0
"""
with open("average_film.in","w") as f:
    f.write(lines)
!/content/q-e/bin/average.x<average_film.in
import numpy as np
%matplotlib inline
import  matplotlib.pyplot as plt

film_avg=13.6*np.loadtxt('avg.dat')
plt.plot(film_avg[:,0],film_avg[:,1])

In [ ]:
lines="""&inputPP
   outdir='interface',
   prefix='RELAX',
   plot_num=11,
   filplot = 'interface.pot'
/

&plot
   filplot(1) = 'interface.pot'
   iflag=3,
   output_format=5
/
"""
with open("pp_interface.in","w") as f:
    f.write(lines)
!/content/q-e/bin/pp.x<pp_interface.in

In [ ]:
lines="""1
interface.pot
1.0
25
3
2.0
"""
with open("average_interface.in","w") as f:
    f.write(lines)
!/content/q-e/bin/average.x<average_interface.in
import numpy as np
%matplotlib inline
import  matplotlib.pyplot as plt

interface_avg=13.6*np.loadtxt('avg.dat')
plt.plot(interface_avg[:,0],interface_avg[:,1])


In [ ]:
lines="""&inputPP
   outdir='subs',
   prefix='RELAX',
   plot_num=11,
   filplot = 'subs.pot'
/

&plot
   filplot(1) = 'subs.pot'
   iflag=3,
   output_format=5,
   e1(1) = 0.0, e1(2) = 0.0, e1(3) = 1.0,
   fileout = 'avg_z_subs.dat',
/
"""
with open("pp_subs.in","w") as f:
    f.write(lines)
!/content/q-e/bin/pp.x<pp_subs.in

In [ ]:
lines="""1
subs.pot
1.0
25
3
2.0
"""
with open("average_subs.in","w") as f:
    f.write(lines)
!/content/q-e/bin/average.x<average_film.in
import numpy as np
%matplotlib inline
import  matplotlib.pyplot as plt

subs_avg=13.6*np.loadtxt('avg.dat')
plt.plot(subs_avg[:,0],subs_avg[:,1])

In [ ]:
from jarvis.core.atoms import Atoms
from jarvis.io.vasp.inputs import Poscar
from intermat.generate import InterfaceCombi
import numpy as np


film_pos = """Si2
1.0
3.3641499856336465 -2.5027128e-09 1.94229273881412
1.121382991333525 3.1717517190189715 1.9422927388141193
-2.5909987e-09 -1.8321133e-09 3.884586486670313
Si
2
Cartesian
3.92483875 2.77528125 6.7980237500000005
0.56069125 0.39646875 0.9711462500000001
"""
subs_pos = """GaAs
1.0
3.509895098013499 -6.584084e-10 2.026439508259945
1.169965032050413 3.3091606691404545 2.026439508259945
-1.8622597e-09 -1.3168164e-09 4.052878022970947
Ga As
1 1
Cartesian
0.0 0.0 0.0
1.1699675 0.82729 2.02644
"""



film = Poscar.from_string(film_pos).atoms
subs = Poscar.from_string(subs_pos).atoms

x = InterfaceCombi(
    film_mats=[film],
    subs_mats=[subs],
    film_indices=[[1,1,0]],
    subs_indices=[[1,1,0]],

    disp_intvl=0.1,
    vacuum_interface=2,
    # dataset=None,
    from_conventional_structure_film=True,
    from_conventional_structure_subs=True,
)
wads = x.calculate_wad(method="ewald")
index = np.argmin(wads) # important for disp_intvl not equal zero
combined = Atoms.from_dict(x.generated_interfaces[index]["generated_interface"])
film_interface = Atoms.from_dict(x.generated_interfaces[index]["film_surf"]).center(vacuum=6)
subs_interface = Atoms.from_dict(x.generated_interfaces[index]["subs_surf"]).center(vacuum=6)


In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=film_interface,
    extra_params=params,
    jobname="film2",
)
out_film= calc.predict()
en = out_film["total_energy"]
print(en)

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=subs_interface,
    extra_params=params,
    jobname="subs2",
)
out_subs= calc.predict()
en = out_subs["total_energy"]
print(en)

In [ ]:
%%time
from intermat.config import IntermatConfig
from intermat.calculators import Calc
params = IntermatConfig().dict()
qe_cmd = "/content/q-e/bin/pw.x"
params['kp_length'] = 0
params['qe_params']['qe_cmd'] = qe_cmd

method = "qe"
calc = Calc(
    method=method,
    atoms=combined,
    extra_params=params,
    jobname="interface2",
)
out_int= calc.predict()
en = out_int["total_energy"]
print(en)

In [ ]:
lines="""&inputPP
   outdir='interface2',
   prefix='RELAX',
   plot_num=11,
   filplot = 'interface2.pot'
/

&plot
   filplot(1) = 'interface2.pot'
   iflag=3,
   output_format=5
/
"""
with open("pp_interface2.in","w") as f:
    f.write(lines)
!/content/q-e/bin/pp.x<pp_interface2.in
lines="""1
interface2.pot
1.0
25
3
2.0
"""
with open("average_interface2.in","w") as f:
    f.write(lines)
!/content/q-e/bin/average.x<average_interface2.in
import numpy as np
%matplotlib inline
import  matplotlib.pyplot as plt

interface_avg=13.6*np.loadtxt('avg.dat')
plt.plot(interface_avg2[:,0],interface_avg2[:,1])


In [ ]:
import numpy as np
%matplotlib inline
import  matplotlib.pyplot as plt
!rm avg.dat
lines="""&inputPP
   outdir='interface2',
   prefix='RELAX',
   plot_num=11,
   filplot = 'interface2.pot'
/

&plot
   filepp(1) = 'interface2.pot'
   iflag=3,
   output_format=5,
   fileout = 'interface2.xsf',
/
"""
with open("pp_interface2.in","w") as f:
    f.write(lines)
!/content/q-e/bin/pp.x<pp_interface2.in
lines="""1
interface2.pot
1.0
25
3
2.0
"""
with open("average_interface2.in","w") as f:
    f.write(lines)



In [ ]:
!/content/q-e/bin/average.x<average_interfac2e.in


interface_avg2=13.6*np.loadtxt('avg.dat')
plt.plot(interface_avg2[:,0],interface_avg2[:,1])

In [ ]:
!ls -altr

In [ ]:
# def read_pot(filename="film.pot"):
#     f=open(filename,"r")
#     lines = f.readlines()
#     f.close()
#     line = lines[1].split()
#     grid=[int(line[0]),int(line[1]),int(line[2])]
#     natoms=int(line[6])
#     species=int(line[7])
#     header=7
#     skip_lines=header+natoms+species
#     chg=[]
#     for line in lines[skip_lines:]:
#         for j in line.split('/n')[0].split():
#           chg.append(float(j))
#     chg=np.array(chg).reshape(grid)
#     return chg
def read_pot(filename="film.pot"):
    f=open(filename,"r")
    lines = f.readlines()
    f.close()
    line = lines[1].split()
    grid=[int(line[0]),int(line[1]),int(line[2])]
    grid_real=[int(line[0])+1,int(line[1])+1,int(line[2])+1]
    print("grid ", grid, " ", grid_real)
    natoms=int(line[6])
    species=int(line[7])
    header=7
    skip_lines=header+natoms+species
    chg=[]
    for line in lines[skip_lines:]:
        for j in line.split('/n')[0].split():
          chg.append(float(j))
          #chg=np.array(chg).reshape(grid)
    print
    print(len(chg))
    print(np.prod(grid))
    print(np.prod(grid_real))
    print
    chg_arr = np.zeros( (grid_real[0], grid_real[1], grid_real[2]))
    c = 0
    for k  in range(grid[2]):
        for j  in range(grid[1]):
            for i  in range(grid[0]):
                #print(i,j,k,c)
                chg_arr[i,j,k] = chg[c]
                c +=1

    for k  in range(grid_real[2]):
        for j  in range(grid_real[1]):
            for i  in range(grid_real[0]):
                chg_arr[i,j,k] = chg_arr[i%grid[0], j%grid[1], k%grid[2]]
    return chg_arr

film_pot = read_pot(filename="film.pot")
subs_pot = read_pot(filename="subs.pot")
interface_pot = read_pot(filename="interface.pot")


In [ ]:

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from jarvis.io.qe.outputs import DataFileSchema
hartree_to_ev = 27.2113839
ry_to_ev = 13.605692
d=DataFileSchema("film/RELAX.save/data-file-schema.xml")
film_efermi = d.efermi
film_mean = ry_to_ev*(np.mean(film_pot,tuple([1,2]))) -film_efermi
print('WF',max(film_mean) )
plt.axhline(y=0,c='g')
plt.axhline(y=np.max(film_mean),c='orange')
plt.xlabel('z ($\AA$)')
plt.ylabel('Potential (eV)')

plt.plot(film_mean)

In [ ]:

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
d=DataFileSchema("subs/RELAX.save/data-file-schema.xml")
subs_efermi = d.efermi
subs_mean = 1*np.mean(subs_pot,tuple([1,2]))-subs_efermi
print('WF',max(subs_mean) )
plt.axhline(y=0,c='g')
plt.axhline(y=np.max(subs_mean),c='orange')
plt.xlabel('z ($\AA$)')
plt.ylabel('Potential (eV)')
plt.plot(subs_mean)


In [ ]:
!ls -altr

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
interface_mean = ry_to_ev*np.mean(interface_pot,tuple([1,2]))
x_val = np.arange(len(interface_mean))
plt.plot(interface_mean)


In [ ]:
from intermat.analyze import *
from scipy.interpolate import CubicSpline
from scipy.signal import find_peaks
def offset(fname="", x=[], s=[], width=5, left_index=-1, polar=False):
    """Get valence band offset."""

    if polar is None:
        polar = check_inerface_polar(fname)
    print("Check polar", polar)
    deltaE = 0 #delta_E(fname)
    S = CubicSpline(x, s)

    max_peaks, properties = find_peaks(s, prominence=1, width=width)
    max_peaks = max_peaks[:-1]

    print("Number of peaks ", len(max_peaks))
    if left_index == -1:  # automatically pick left_index from max_peaks
        print("auto detect left index")
        if len(max_peaks) <= 8:
            print("WARNING, not many peaks found")
            left_index = 1
        elif len(max_peaks) <= 12:
            left_index = 2
        else:
            left_index = 3
    else:
        print("use input left index")
    print("left index ", left_index)
    right_index = left_index * -1 + 1

    plt.plot(x[max_peaks], s[max_peaks], "x")
    # tmp=int((max_peaks[left_index]-max_peaks[left_index+2]))
    x_target1 = x[
        np.arange(max_peaks[left_index], max_peaks[left_index + 1], 2)
    ]

    # initial guess left
    L_guess_peaks_left = x_target1[-1] - x_target1[0]

    # points in left cell
    # x_target1 = x[ range(50, 100,2) ]
    print("Initial guess left ", L_guess_peaks_left)
    L = best_L_recursive(1.0, L_guess_peaks_left * 1.5, S, x_target1)
    print("Lleft ", L)

    plt.plot(x, s, c="k")
    XX, AVG = do_average(L, x, S)

    meanval1, m1, c1 = get_mean_val(x_target1, XX, AVG)
    if polar:
        plt.plot(XX, np.array(XX) * m1 + c1, c="purple", linestyle="-.")
    # print('x_target1',x_target1)
    # print('XX',XX)
    # print('AVG',AVG)
    plt.plot(XX, AVG, c="r")

    # x_target2 = x[-200:-150]
    # tmp=int((max_peaks[right_index-1]-max_peaks[right_index]))
    x_target2 = x[
        np.arange(max_peaks[right_index - 1], max_peaks[right_index], 2)
    ]
    # x_target2 = x[ range(-100, ,2) ]

    # initial guess right
    L_guess_peaks_right = x_target2[-1] - x_target2[0]
    L = best_L_recursive(1.0, L_guess_peaks_right * 1.5, S, x_target2)
    print("Initial guess right ", L_guess_peaks_right)
    print("Lright ", L)

    plt.plot(x, s, c="k")
    XX, AVG = do_average(L, x, S)
    meanval2, m2, c2 = get_mean_val(x_target2, XX, AVG)
    if polar:
        plt.plot(XX, np.array(XX) * m2 + c2, c="orange", linestyle="-.")

    if polar:
        mid_point = int((len(XX) - 1) / 2)
        polar_del_V = (np.array(XX) * m2 + c2)[mid_point] - (
            np.array(XX) * m1 + c1
        )[mid_point]
        plt.plot(mid_point, -7, "*")
        plt.axvline(x=XX[mid_point], linestyle="-.", c="blue")
        print(
            "polar delV,mid_point",
            polar_del_V,
            mid_point,
            len(XX),
            XX[mid_point],
        )

    plt.plot(XX, AVG, c="g")
    plt.plot(x_target1, S(x_target1), "c")
    plt.plot(x_target2, S(x_target2), "c")
    deltaV = meanval2 - meanval1
    phi = deltaV + deltaE
    if polar:
        phi = polar_del_V + deltaE
        deltaV = polar_del_V
    plt.grid(color="gray", ls="-.")
    plt.minorticks_on()
    plt.ylabel("Potential (eV)")
    plt.xlim(0, max(x))
    plt.xlabel(r"Distance ($\AA$)")
    # plt.plot(x,[meanval1 for i in range(len(x))],linestyle='-.',color='blue')
    # plt.plot(x,[meanval2 for i in range(len(x))],linestyle='-.',color='blue')
    print("meanval ", [meanval1, meanval2], meanval2 - meanval1, phi)
    plt.title("Offset (eV): " + str(round(phi, 2)))
    filename = "offset_max-" + fname.split("/")[0] + ".png"
    print("deltaE", deltaE)
    # plt.show()
    plt.savefig(filename)
    plt.close()
    info = {}
    info["phi"] = phi
    info["polar"] = polar
    info["deltaV"] = deltaV
    info["deltaE"] = deltaE
    info["L_guess_peaks_right"] = L_guess_peaks_right
    info["L"] = L
    info["left_index"] = left_index

    return info  # phi


offset(x=x_val,s=interface_mean,)

In [ ]:
from jarvis.io.qe.outputs import DataFileSchema
d=DataFileSchema("film/RELAX.save/data-file-schema.xml")
d.data['qes:espresso']['output']['basis_set']

In [ ]:
import pprint
nr1=int(d.data['qes:espresso']['output']['basis_set']['fft_grid']['@nr1'])
nr2=int(d.data['qes:espresso']['output']['basis_set']['fft_grid']['@nr2'])
nr3=int(d.data['qes:espresso']['output']['basis_set']['fft_grid']['@nr3'])

In [ ]:
d.data['qes:espresso']['output']['basis_set']

In [ ]:
59987/36

In [ ]:
nr1,nr2,nr3

In [ ]:
data.shape

In [ ]:
nr1*nr2*nr3

In [ ]:
839876/186624

In [ ]:
d.data['qes:espresso']['output']['basis_set']